# 05 - 训练基础**学习目标：**- 理解 YOLO 的 Loss 函数组成- 理解优化器和学习率调度- 学会解读训练曲线- 在 COCO128 上完成第一次训练---

## 1. 训练流程```初始化模型权重（随机或预训练）    │    ▼┌──────────────────────────────────────┐│ for each epoch:                      ││   for each batch:                    ││     1. 前向传播 → 得到预测            ││     2. 计算 Loss（预测 vs 真实标注）  ││     3. 反向传播 → 计算梯度            ││     4. 更新权重（优化器）             ││   验证集评估                          ││   保存最优模型                        │└──────────────────────────────────────┘```

## 2. Loss 函数详解YOLOv8 的 Loss 由 3 部分组成：```Total Loss = box_loss + cls_loss + dfl_loss```| Loss | 全称 | 作用 | 说明 ||------|------|------|------|| **box_loss** | CIoU Loss | 框的位置准确性 | 预测框与真实框的 IoU || **cls_loss** | BCE Loss | 分类准确性 | 类别预测的交叉熵 || **dfl_loss** | Distribution Focal Loss | 框的边界精确度 | 回归框边界的分布 |**直觉理解**：- box_loss 高 → 框的位置偏了- cls_loss 高 → 类别判断错了- dfl_loss 高 → 框的边界不够精确

## 3. 优化器和学习率**常用优化器**：- **SGD** (随机梯度下降)：经典，需要仔细调参- **AdamW**：自适应学习率，通常更稳定**学习率调度**：```学习率  │  │   ┌──warmup──┐  │   /          \  │  /            \___cosine annealing___  │ /                                      \  └──────────────────────────────────────────→ epoch  0                                        100```- **Warmup**: 前几轮用小学习率，避免初期震荡- **Cosine Annealing**: 学习率按余弦曲线下降

## 4. 动手实验：在 COCO128 上训练

In [ ]:
# 教学演示 — 下面的实现帮助理解原理，生产代码请用 yolo_learn.*
from ultralytics import YOLO# 加载预训练模型model = YOLO("yolo11n.pt")# 在 COCO128 上训练results = model.train(    data="../configs/data/coco128.yaml",    epochs=10,        # 先跑 10 轮看看效果    imgsz=640,        # 输入图片尺寸    batch=16,         # batch size    device="mps",     # Apple Silicon    project="../outputs",    name="first_train",    verbose=True,)

## 5. 解读训练曲线

In [ ]:
# 生产路径：使用 yolo_learn 封装
from yolo_learn.models.train import train

# train() 自动管理输出目录和配置快照
# results = train(
#     model="yolo11n.pt",
#     data="../configs/data/coco128.yaml",
#     epochs=10,
#     name="first_train",
# )


In [ ]:
import matplotlib.pyplot as pltimport pandas as pdfrom pathlib import Path# 读取训练日志log_dir = Path("../outputs/first_train")csv_path = log_dir / "results.csv"if csv_path.exists():    df = pd.read_csv(csv_path, skipinitialspace=True)        fig, axes = plt.subplots(2, 2, figsize=(14, 10))        # Loss 曲线    axes[0,0].plot(df["epoch"], df["train/box_loss"], label="box_loss")    axes[0,0].plot(df["epoch"], df["train/cls_loss"], label="cls_loss")    axes[0,0].plot(df["epoch"], df["train/dfl_loss"], label="dfl_loss")    axes[0,0].set_xlabel("Epoch")    axes[0,0].set_ylabel("Loss")    axes[0,0].set_title("Training Loss")    axes[0,0].legend()    axes[0,0].grid(True, alpha=0.3)        # mAP 曲线    axes[0,1].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.5")    axes[0,1].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")    axes[0,1].set_xlabel("Epoch")    axes[0,1].set_ylabel("mAP")    axes[0,1].set_title("Validation mAP")    axes[0,1].legend()    axes[0,1].grid(True, alpha=0.3)        # 学习率    axes[1,0].plot(df["epoch"], df["lr/pg0"], label="lr")    axes[1,0].set_xlabel("Epoch")    axes[1,0].set_ylabel("Learning Rate")    axes[1,0].set_title("Learning Rate Schedule")    axes[1,0].grid(True, alpha=0.3)        # Precision & Recall    if "metrics/precision(B)" in df.columns:        axes[1,1].plot(df["epoch"], df["metrics/precision(B)"], label="Precision")        axes[1,1].plot(df["epoch"], df["metrics/recall(B)"], label="Recall")        axes[1,1].set_xlabel("Epoch")        axes[1,1].set_ylabel("Score")        axes[1,1].set_title("Precision & Recall")        axes[1,1].legend()        axes[1,1].grid(True, alpha=0.3)        plt.tight_layout()    plt.show()else:    print(f"训练日志未找到: {csv_path}")    print("请先完成 05 节的训练")

## 6. 超参数的影响| 参数 | 作用 | 太小的影响 | 太大的影响 ||------|------|-----------|-----------|| **epochs** | 训练轮数 | 欠拟合 | 过拟合、浪费时间 || **batch** | 每批图片数 | 梯度不稳定 | 显存不足 || **lr0** | 初始学习率 | 收敛慢 | 震荡、不收敛 || **imgsz** | 输入尺寸 | 小目标丢失 | 显存大、速度慢 |**在 MPS 上的建议配置**：```yamlepochs: 30-50batch: 8-16imgsz: 640device: mps```

## 7. 练习### 练习 1: 调整训练轮数把 epochs 改为 30，观察 mAP 是否继续提升。### 练习 2: 尝试不同 batch size分别用 batch=8 和 batch=16 训练，对比训练速度和最终效果。### 练习 3: 对比预训练 vs 从零训练用 `model.train(pretrained=False)` 从随机权重训练，对比效果差距。

## 📝 学习检查

加载本章检查点和练习，检验学习效果：

```python
from yolo_learn.pedagogy.checkpoint import Quiz
from yolo_learn.pedagogy.scaffold import ExerciseSet
from yolo_learn.pedagogy.reflection import ReflectionSet, MistakeLibrary
```


In [ ]:
# 加载本章检查点
quiz = Quiz.from_yaml("05_training_basics")
print(f"本章 {quiz.total} 个检查点 + 预测练习")

# 查看第一题
print(quiz.ask(0))


### ✍️ 练习


In [ ]:
# 加载本章练习
exercises = ExerciseSet.from_yaml("05_training_basics")
print(f"本章 {exercises.total} 个练习")
for ex in exercises.exercises:
    print()
    print(ex.show())


### 🤔 反思与自评


In [ ]:
# 加载反思问题和自评量表
reflections = ReflectionSet.from_yaml("05_training_basics")
print(reflections.show_reflections())
print()
print(reflections.show_assessments())

# 常见错误
mistakes = MistakeLibrary()
print()
print(mistakes.show_for_topic("training_basics"))


---**上一节**: [04 - 推理与预测](04_inference_with_pretrained.ipynb)**下一节**: [06 - 迁移学习](06_transfer_learning.ipynb) - 用预训练模型微调